<a href="https://colab.research.google.com/github/iamrobby/MedTriage/blob/main/MedTriage.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install Groq
!pip install faiss-cpu
!pip install sentence-transformers
!pip install streamlit pyngrok nest_asyncio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 61.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 98.2 MB/s eta 0:00:00


In [ ]:
%%writefile triage_backend.py
import faiss
from sentence_transformers import SentenceTransformer
import numpy as np
from groq import Groq
def remove_old_info(text):

    old_markers = [
        "years ago", "in 19", "childhood",
        "long ago"
    ]

    #  REMOVE "history of"
    # it's dangerous

    lines = text.split("\n")
    filtered = []

    for line in lines:
        if not any(marker in line.lower() for marker in old_markers):
            filtered.append(line)

    return " ".join(filtered)
def extract_symptoms(text):

    symptoms = [
        "chest pain", "breathing difficulty", "shortness of breath",
        "fever", "vomiting", "bleeding", "dizziness",
        "unconscious", "headache"
    ]

    found = []

    text = text.lower()

    for s in symptoms:
        if s in text:
            found.append(s)

    return found
def filter_relevant(text):

    keywords = [
        "pain", "breath", "shortness", "tightness",
        "pressure", "dizziness", "sweating",
        "nausea", "weakness", "speech",
        "bleeding", "unconscious"
    ]

    sentences = text.split(".")
    filtered = []

    for s in sentences:
        if any(k in s.lower() for k in keywords):
            filtered.append(s.strip())

    #  MOST IMPORTANT LINE
    if not filtered:
        return text   # NEVER return empty

    return ". ".join(filtered)
def build_query(symptoms, clean_context):

    if not symptoms:
        return clean_context   #  fallback to full context

    return ", ".join(symptoms)
def context_pruning_pipeline(raw_input):

    step1 = remove_old_info(raw_input)

    step2 = filter_relevant(step1)
    if not step2.strip():
        step2 = raw_input

    symptoms = extract_symptoms(step2)

    final_query = build_query(symptoms,step2)

    return {
        "clean_text": step2,
        "symptoms": symptoms,
        "final_query": final_query
    }
pruned = context_pruning_pipeline("Patient is a 58-year-old male presenting to the emergency department with complaints of persistent chest discomfort for the past 2 hours. He describes the pain as a tightness or pressure in the center of his chest, occasionally radiating to his left arm and jaw. The patient also reports shortness of breath, especially when lying down, and a feeling of dizziness. He mentions that he was sweating profusely earlier and felt nauseous but did not vomit.The patient denies any recent trauma, fever, or cough but says he has been feeling unusually fatigued over the past few days. No known history of similar episodes in the past. Family history reveals that his father had a heart attack at the age of 60.On observation, the patient appears anxious and slightly pale. No visible external injuries. He is able to speak in full sentences but prefers to remain seated due to discomfort. Additional notes: The patient also mentioned mild back pain earlier in the morning, which he initially ignored. No known allergies. No recent travel history.")
clean_context = pruned["clean_text"]
question = pruned["final_query"]
symptoms = pruned["symptoms"]
def trim_context(text, max_words=100):
    words = text.split()
    return " ".join(words[:max_words])
clean_context = trim_context(clean_context)

# Initialize embedding model
embed_model = SentenceTransformer('all-MiniLM-L6-v2')  # fast & small

# Sample emergency protocols (can expand later)
protocols_data = [

    {
        "condition": "Myocardial Infarction",
        "symptoms": ["chest pain", "radiating arm pain", "sweating", "nausea"],
        "protocol": "Possible heart attack. Immediate ECG, cardiac monitoring, and emergency intervention required.",
        "severity": "RED"
    },

    {
    "condition": "Stroke (CVA)",
    "symptoms": [
        "slurred speech",
        "facial drooping",
        "one sided weakness",
        "sudden onset",
        "difficulty speaking",
        "arm weakness"
    ],
    "protocol": "Sudden onset of slurred speech, facial drooping, and one-sided weakness strongly indicates stroke (CVA). Immediate CT scan required within golden hour. Activate emergency stroke protocol.",
    "severity": "RED"
},


    {
        "condition": "Anaphylaxis",
        "symptoms": ["swelling", "difficulty breathing", "allergy exposure"],
        "protocol": "Severe allergic reaction. Administer epinephrine immediately. Secure airway.",
        "severity": "RED"
    },

    {
        "condition": "Pulmonary Embolism",
        "symptoms": ["chest pain", "shortness of breath", "rapid heart rate"],
        "protocol": "Possible pulmonary embolism. Oxygen support and urgent imaging required.",
        "severity": "RED"
    },

    {
        "condition": "Seizure",
        "symptoms": ["convulsions", "loss of consciousness"],
        "protocol": "Protect patient from injury. Maintain airway. Monitor vitals.",
        "severity": "YELLOW"
    },

    {
        "condition": "Fracture",
        "symptoms": ["bone pain", "swelling", "deformity"],
        "protocol": "Immobilize affected area. Provide pain management.",
        "severity": "YELLOW"
    }
]

# Generate embeddings
texts = [
    f"Condition: {item['condition']}. Symptoms: {' '.join(item['symptoms'])}. Protocol: {item['protocol']}"
    for item in protocols_data
]

embeddings = embed_model.encode(texts, convert_to_numpy=True)

# Initialize FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)  # L2 distance
index.add(embeddings)  # Add vectors to index
def retrieve_context(query, top_k=3):

    query_embedding = embed_model.encode([query])
    distances, indices = index.search(query_embedding, top_k)

    results = [protocols_data[i] for i in indices[0]]

    formatted = [
        f"Condition: {r['condition']}\nProtocol: {r['protocol']}"
        for r in results
    ]

    return "\n\n".join(formatted)
headers = {
    "x-api-key": "SCALEDOWN_KEY",
    "Content-Type": "application/json"
}
import requests

def compress_context(context, question):

    url = "https://api.scaledown.xyz/compress/raw/"

    payload = {
        "context": context,
        "prompt": question,
        "scaledown": {"rate": "auto"}
    }

    response = requests.post(url, headers=headers, json=payload)

    try:
        data = response.json()

        if data.get("successful") and "results" in data:
            return data["results"].get("compressed_prompt", context)

        else:
            print("⚠️ API returned unexpected structure:", data)
            return context

    except Exception as e:
        print("⚠️ API parsing error:", e)
        return context
client = Groq(api_key="GROQ_KEY")

def ask_llm(context, question):

    prompt = f"""
    You are an emergency triage assistant.

    Context:
    {context}

    Question:
    {question}

    Give a short emergency assessment (4 lines max).
    """

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0.3,
        max_tokens=150
    )

    return response.choices[0].message.content
print(ask_llm(
    "Patient has chest pain and shortness of breath",
    "What is the likely emergency and What needs to be done Immediately?"
))
def run_system(raw_input):

    pruned = context_pruning_pipeline(raw_input)

    clean_context = pruned["clean_text"]
    question = pruned["final_query"]
    symptoms = pruned["symptoms"]

    retrieved_context = retrieve_context(clean_context + " " + raw_input)
    retrieved_context=trim_context(retrieved_context)

    combined_context = clean_context + "\n" + retrieved_context
    compressed_context = compress_context(combined_context, question)
    answer = ask_llm(compressed_context, question)

    return answer

Overwriting triage_backend.py


In [ ]:
%%writefile app.py
import streamlit as st
from triage_backend import run_system

st.set_page_config(
    page_title="AI Emergency Triage",
    layout="centered"
)

st.title("AI Emergency Response Triage System")

st.markdown(
"""
Enter patient symptoms or emergency description.
The AI will analyze severity and suggest triage action.
"""
)

user_input = st.text_area(
    "Patient Description",
    height=200,
    placeholder="Example: 58-year-old male with chest pain, sweating, dizziness..."
)

if st.button("Run Triage Analysis"):

    if user_input.strip() == "":
        st.warning("Please enter patient details.")
    else:
        with st.spinner("Analyzing emergency case..."):

            result = run_system(user_input)

        st.success("Analysis Complete")

        st.subheader("🩺 Triage Decision")
        st.write(result)

In [ ]:
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

In [ ]:
!streamlit run app.py &>/content/log.txt &

In [ ]:
!cloudflared tunnel --url http://localhost:8501